# NLP Job Matcher: Colab smoke run
Select **Runtime → Change runtime type → GPU**. This notebook is an unexecuted walkthrough; local execution evidence is in the reports.

The updated setup files must be present in the GitHub repository before cloning. Otherwise upload a source ZIP containing this setup and use that folder. No credentials or paid API are needed for the public assets. The two-step run uses out-of-scope proxy data solely to verify training mechanics.


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/NLP_Job_Matcher')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WillBranam/NLP_Job_Matcher.git', str(repo)], check=True)
assert (repo / 'scripts/train_lora.py').exists(), 'Upload or push the setup files first.'
%cd /content/NLP_Job_Matcher


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
# If Colab asks for a runtime restart after dependency changes, restart and resume here.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'
print(torch.cuda.get_device_name(0))


In [ ]:
subprocess.run([sys.executable, 'scripts/download_assets.py'], check=True)
subprocess.run([sys.executable, 'scripts/prepare_data.py'], check=True)


In [ ]:
subprocess.run([sys.executable, 'scripts/infer.py', '--device', 'cuda'], check=True)
subprocess.run([sys.executable, 'scripts/train_lora.py', '--config', 'configs/smoke.yaml', '--device', 'cuda'], check=True)
subprocess.run([sys.executable, 'scripts/infer.py', '--adapter', 'models/adapters/smoke', '--output', 'outputs/predictions/smoke_adapter_example.json', '--device', 'cuda'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Save evidence before the Colab session ends
Download the archive from the Files pane. It contains outputs, adapter weights and tokenizer, configuration, and installed package versions. It intentionally excludes the multi-gigabyte base weights and source datasets, which can be downloaded again. Inspect `valid_schema` in both prediction files: generated text is retained even when it fails validation.

A successful smoke run does not establish accuracy or improvement. Review `reports/MidtermChecklist.md` and add genuine in-scope data before full training. Colab execution has not been verified by local setup.


In [ ]:
import shutil
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, stdout=open('outputs/colab_packages.txt', 'w'))
with __import__('zipfile').ZipFile('/content/midterm_evidence.zip', 'w', compression=__import__('zipfile').ZIP_DEFLATED) as archive:
    for folder in ['outputs', 'models/adapters/smoke', 'configs']:
        for path in Path(folder).rglob('*'):
            if path.is_file():
                archive.write(path, str(path))
print('Saved /content/midterm_evidence.zip')
